# Python 面向对象高级详解

> **一句话本质**：Python 面向对象的高级之处不在于继承，而在于**数据模型协议**——通过魔术方法、描述符、元类，让自定义对象像内置对象一样自然地融入语言。

**业务场景说明**：本 Notebook 的示例围绕 **LLMOps API 服务开发** 取材——即大模型应用的配置管理、调用参数校验、多 Provider（模型供应商）插件化接入、调用耗时与事件观测。这是对需求占位符 `{{业务场景描述}}` 的合理假设，如需替换为其他场景，整体结构不变，仅替换示例素材。

## 学习目标

1. 掌握魔术方法（`__new__`/`__init__`/`__del__`、比较、算术、容器、可调用、上下文、属性访问）
2. 理解 `property`、`cached_property` 与 `__getattr__` 的分工
3. 吃透描述符协议（数据/非数据描述符、`__set_name__`、属性查找优先级）
4. 掌握 MRO（C3 线性化）、`super()` 的真实语义、协作式多继承与 Mixin
5. 会用抽象基类（ABC）定义接口约束，理解虚拟子类
6. 理解元类的原理与边界，掌握 `__init_subclass__` / `__set_name__` 等轻量替代
7. 熟练使用 `dataclass`、`NamedTuple`、`__slots__` 减少样板代码并优化内存
8. 能把上述机制组合成工程级方案：校验描述符、插件注册、单例配置、观察者、缓存代理

## 阅读方式

- 每个 Markdown 单元讲原理，紧跟的 Code 单元是**真实执行过**的可运行示例（输出已嵌入）
- 反例单元：先触发真实报错（用 `try/except` 捕获并打印，保证整个 Notebook 0 报错），再给出正确写法对比
- 涉及「执行时机 / 顺序 / 等价性」的论断均配有验证代码

# 第 0 章 前置知识

**本章解决什么问题**：面向对象高级特性全部建立在几个底层事实上——实例字典、类字典、`type` 元类、属性查找顺序。先把这四件事用代码验证清楚，后面的描述符、元类才不会是空中楼阁。

In [1]:
# ① 前置知识 1：实例属性与类属性存放于不同的字典
class User:
    type = "member"          # ② 类属性：存放在【类】的 __dict__ 中，所有实例共享

u1, u2 = User(), User()      # ③ 创建两个实例
u1.name = "Alice"            # ④ 实例属性：存放在【实例自己的】__dict__ 中

print("类字典  :", User.__dict__["type"])      # ⑤ 类字典里有 type
print("实例字典:", u1.__dict__)                  # ⑥ 实例字典里只有 name
print("u2 字典 :", u2.__dict__)                  # ⑦ u2 没有设置任何实例属性

# ⑧ 验证「共享」：通过类修改类属性，所有实例可见
User.type = "admin"
print("修改后 u1.type =", u1.type, "| u2.type =", u2.type)

类字典  : member
实例字典: {'name': 'Alice'}
u2 字典 : {}
修改后 u1.type = admin | u2.type = admin


In [2]:
# ① 前置知识 2：type(obj) 是类，类的类是 type —— 这就是元类的伏笔
class User:
    pass

u = User()
print("type(u)     =", type(u))       # ② 实例的类型是类 User
print("type(User)  =", type(User))    # ③ 类的类型是 type（User 由 type 创建）
print("type(type)  =", type(type))    # ④ type 的类型还是 type（链条终点）

# ⑤ 等价性验证：class 语句只是 type(...) 的语法糖
User2 = type("User2", (), {"name": "Alice"})   # ⑥ 三参数 type：类名、父类元组、命名空间
print("User2().name =", User2().name)
print("等价性: type(User2) is type =", type(User2) is type)

type(u)     = <class '__main__.User'>
type(User)  = <class 'type'>
type(type)  = <class 'type'>
User2().name = Alice
等价性: type(User2) is type = True


In [3]:
# ① 前置知识 3：属性查找顺序（描述符优先级的前置）
class Demo:
    cls_attr = "类属性"        # ② 类属性

d = Demo()
d.cls_attr = "实例属性"         # ③ 同名实例属性会【遮蔽】类属性
print("d.cls_attr    =", d.cls_attr)          # ④ 命中实例字典
print("Demo.cls_attr =", Demo.cls_attr)       # ⑤ 类身上仍是类属性
del d.cls_attr                            # ⑥ 删除实例字典中的键，类属性重新可见
print("删除后 d.cls_attr =", d.cls_attr)

d.cls_attr    = 实例属性
Demo.cls_attr = 类属性
删除后 d.cls_attr = 类属性


In [4]:
# ① 前置知识 4：一切皆对象——函数、类都能作为值传递（Mixin/装饰器/观察者的基础）
def shout(text):              # ② 函数是对象
    return text.upper()

fn = shout                    # ③ 函数名只是标签，可以赋给另一个名字
print("函数是一等公民:", fn("hello"))

def apply(fn, arg):           # ④ 函数可以作为参数传入（观察者模式的基础）
    return fn(arg)
print("函数可传递:", apply(shout, "llmops"))

class Adder:                  # ⑤ 类也是对象，可以作为参数传入（插件注册的基础）
    def __init__(self, n):
        self.n = n
    def add(self, x):
        return self.n + x

def create(cls, *args):       # ⑥ 类作为参数传入工厂函数
    return cls(*args)

print("类可传递:", [a.add(10) for a in (create(Adder, 1), create(Adder, 2))])   # ⑦ 11, 12

函数是一等公民: HELLO
函数可传递: LLMOPS
类可传递: [11, 12]


# 第 1 章 魔术方法（Magic Methods）

**本章解决什么问题**：让自定义对象像 `list`、`int`、`str` 一样参与 `+`、`==`、`len()`、`for`、`with`、`()` 等语言级操作——这是 Python 数据模型协议的核心。

魔术方法（dunder methods，双下划线方法）由 **Python 解释器在特定操作时自动调用**，你几乎从不直接调用它们。

| 类别 | 代表方法 | 触发时机 |
|---|---|---|
| 生命周期 | `__new__` / `__init__` / `__del__` | 创建 / 初始化 / 回收 |
| 字符串表示 | `__repr__` / `__str__` | `repr()` / `print()` |
| 比较运算 | `__eq__` / `__lt__` / `__hash__` | `==` / `<` / `hash()` |
| 算术运算 | `__add__` / `__mul__` / `__rmul__` | `+` / `*` |
| 容器协议 | `__len__` / `__getitem__` / `__iter__` | `len()` / `[]` / `for` |
| 可调用 | `__call__` | `obj(...)` |
| 上下文管理 | `__enter__` / `__exit__` | `with` |
| 属性访问 | `__getattr__` / `__setattr__` | 属性读缺失 / 属性写 |

## 1.1 对象生命周期：`__new__` → `__init__` → `__del__`

- `__new__`：**类方法**，负责创建并返回实例（先于 `__init__` 执行）
- `__init__`：初始化实例（此时实例已存在）
- `__del__`：对象被垃圾回收时调用，**不保证时机**，绝不能依赖它释放资源

In [5]:
# ① 生命周期三兄弟的执行顺序验证
import gc                     # ② 手动触发垃圾回收，让 __del__ 的输出变得确定

class Demo:
    def __new__(cls, *args, **kwargs):
        print("1. __new__  -> 创建实例")            # ③ 第一步：分配内存、返回实例
        return super().__new__(cls)                  # ④ 必须返回实例，否则 __init__ 不会执行

    def __init__(self, x):
        print("2. __init__ -> 初始化实例")           # ⑤ 第二步：拿到 __new__ 返回的实例做初始化
        self.x = x

    def __del__(self):
        print("3. __del__  -> 对象被回收")           # ⑥ 时机不保证；这里用 gc.collect() 强制触发

d = Demo(1)
print("使用对象:", d.x)
del d                          # ⑦ 删除引用
gc.collect()                   # ⑧ 强制 GC，确保 __del__ 在本单元内被调用（保证输出确定）

1. __new__  -> 创建实例
2. __init__ -> 初始化实例
使用对象: 1
3. __del__  -> 对象被回收


25

`__new__` 最典型的用途：**单例** 与 **不可变类型子类化**（如继承 `int`/`tuple` 时必须在 `__new__` 里传值）。

In [6]:
# ① 单例模式：__new__ 拦截重复创建
class Singleton:
    _instance = None            # ② 类属性保存唯一实例

    def __new__(cls):
        if cls._instance is None:                    # ③ 第一次创建
            cls._instance = super().__new__(cls)     # ④ 调用 object.__new__ 真正分配内存
        return cls._instance                         # ⑤ 之后直接返回缓存实例

a = Singleton()
b = Singleton()
print("a is b =", a is b)       # ⑥ is 验证：同一个对象 → True

a is b = True


## 1.2 字符串表示：`__repr__` 与 `__str__`

| 方法 | 受众 | 约定 | 回退关系 |
|---|---|---|---|
| `__repr__` | 开发者（调试、日志） | 理想情况下 `eval(repr(obj)) == obj` | 无 |
| `__str__` | 最终用户 | 可读性优先 | 未定义时**回退到 `__repr__`** |

In [7]:
# ① 双表示方法的区别与回退关系
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"   # ② 面向开发者：无歧义、可重建

    def __str__(self):
        return f"({self.x}, {self.y})"        # ③ 面向用户：简洁可读

p = Point(1, 2)
print("repr(p) =", repr(p))    # ④ 显式 repr
print("str(p)  =", str(p))     # ⑤ 显式 str
print("print(p)=", p)          # ⑥ print 优先走 __str__

# ⑦ 回退验证：只定义 __repr__ 的类，print 时自动使用 __repr__
class OnlyRepr:
    def __repr__(self):
        return "OnlyRepr<...>"
print("回退验证:", OnlyRepr())

repr(p) = Point(1, 2)
str(p)  = (1, 2)
print(p)= (1, 2)
回退验证: OnlyRepr<...>


## 1.3 比较运算：`__eq__` / `__lt__` / `__hash__`

三条铁律：
1. 与**未知类型**比较时返回 `NotImplemented`（注意：不是 `NotImplementedError` 异常），让 Python 尝试反射方法或报 `TypeError`
2. 定义 `__eq__` 会把 `__hash__` 隐式置为 `None`（对象不可哈希），需要显式定义
3. `__eq__` 与 `__hash__` 必须基于**同一组字段**，否则违反哈希不变约定

In [8]:
# ① 完整的比较协议实现（LLMOps 场景：版本号比较，判断模型 API 版本新旧）
class Version:
    def __init__(self, major, minor):
        self.major = major
        self.minor = minor

    def __eq__(self, other):
        if not isinstance(other, Version):     # ② 未知类型 → 交还解释器处理
            return NotImplemented
        return (self.major, self.minor) == (other.major, other.minor)

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor) < (other.major, other.minor)

    def __hash__(self):                        # ③ 与 __eq__ 基于同一组字段
        return hash((self.major, self.minor))

print("1.2 == 1.2 :", Version(1, 2) == Version(1, 2))   # ④ True
print("1.2 <  2.0 :", Version(1, 2) < Version(2, 0))    # ⑤ True
print("可哈希     :", {Version(1, 2): "v1.2"}[Version(1, 2)])  # ⑥ 值相同的对象哈希也相同

# ⑦ 与未知类型比较 → Python 得到 NotImplemented 后抛 TypeError（而非 AttributeError）
try:
    Version(1, 2) == "1.2"
except TypeError as e:
    print("与非法类型比较 →", type(e).__name__, ":", e)

1.2 == 1.2 : True
1.2 <  2.0 : True
可哈希     : v1.2


`functools.total_ordering` 只需定义 `__eq__` 和 `__lt__`，即可自动补全 `__le__` / `__gt__` / `__ge__`。

> **修正说明 1（源文档勘误）**：源文档中 `@total_ordering` 版本的 `__eq__`/`__lt__` 删除了 `isinstance` 检查。这样当与 `Version` 之外的类型比较时会抛 `AttributeError`（访问 `other.major` 失败），而不是正确的 `TypeError`。下面先复现错误行为，再给出修正版。

In [9]:
# ① 复现源文档的「坏味道」写法：缺少类型检查
from functools import total_ordering

@total_ordering
class BadVersion:
    def __init__(self, major, minor):
        self.major, self.minor = major, minor
    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)  # ② other 没有这些属性
    def __lt__(self, other):
        return (self.major, self.minor) < (other.major, other.minor)

try:
    BadVersion(1, 2) < (2, 0)     # ③ 与元组比较：other.major 抛 AttributeError
except AttributeError as e:
    print("坏味道写法 →", type(e).__name__, ":", e)

坏味道写法 → AttributeError : 'tuple' object has no attribute 'major'


In [10]:
# ① 修正版：total_ordering + 类型检查
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major, minor):
        self.major, self.minor = major, minor

    def __eq__(self, other):
        if not isinstance(other, Version):        # ② 未知类型返回 NotImplemented
            return NotImplemented
        return (self.major, self.minor) == (other.major, other.minor)

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return (self.major, self.minor) < (other.major, other.minor)

    def __hash__(self):                            # ③ 注意：total_ordering 不会补 __hash__
        return hash((self.major, self.minor))

vs = [Version(2, 0), Version(1, 9), Version(1, 2)]
print("自动补全排序:", [f"{v.major}.{v.minor}" for v in sorted(vs)])   # ④ __gt__/__le__ 等由装饰器生成
try:
    Version(1, 2) < (2, 0)
except TypeError as e:
    print("修正后 →", type(e).__name__, ":", e)    # ⑤ 正确的报错类型

自动补全排序: ['1.2', '1.9', '2.0']
修正后 → TypeError : '<' not supported between instances of 'Version' and 'tuple'


## 1.4 算术运算：`__add__` / `__sub__` / `__mul__` / `__rmul__` / `__neg__`

`__rmul__`（反射乘法）解决 `2 * v` 这种**右操作数是自定义对象**的场景：Python 先尝试 `(2).__mul__(v)` 失败后，回头发起 `v.__rmul__(2)`。

In [11]:
# ① 向量算术：让自定义对象参与 + - * 运算
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __add__(self, other):                    # ② v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):                    # ③ v1 - v2
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):                   # ④ v * 2
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):                  # ⑤ 2 * v：左操作数 int 不支持，反射到右操作数
        return self.__mul__(scalar)

    def __neg__(self):                           # ⑥ -v
        return Vector(-self.x, -self.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v = Vector(1, 2) + Vector(3, 4)
print("v1 + v2 =", v)
print("2 * v   =", 2 * v)      # ⑦ 走 __rmul__
print("v * 2   =", v * 2)      # ⑧ 走 __mul__
print("-v      =", -v)

v1 + v2 = Vector(4, 6)
2 * v   = Vector(8, 12)
v * 2   = Vector(8, 12)
-v      = Vector(-4, -6)


## 1.5 容器协议：让对象像 `list` 一样工作

实现 `__len__` + `__getitem__` + `__iter__` 后，对象即可参与 `len()`、索引、`in`、`for`。

In [12]:
# ① 完整容器协议：自定义列表
class MyList:
    def __init__(self):
        self._data = []                          # ② 内部委托给真正的 list（组合优于继承）

    def __len__(self):                           # ③ len(ml)
        return len(self._data)

    def __getitem__(self, index):                # ④ ml[0] / ml[1:3]（切片自动支持）
        return self._data[index]

    def __setitem__(self, index, value):         # ⑤ ml[0] = x
        self._data[index] = value

    def __delitem__(self, index):                # ⑥ del ml[0]
        del self._data[index]

    def __contains__(self, item):                # ⑦ x in ml（不定义时会用 __iter__ 兜底）
        return item in self._data

    def __iter__(self):                          # ⑧ for x in ml
        return iter(self._data)

    def append(self, item):                      # ⑨ 保留一个真实的方法名
        self._data.append(item)

ml = MyList()
ml.append(1)
ml.append(2)
ml[0] = 99                     # ⑩ __setitem__
print("len(ml) =", len(ml))
print("ml[0]   =", ml[0])
print("99 in ml =", 99 in ml)
print("遍历    :", [x for x in ml])

len(ml) = 2
ml[0]   = 99
99 in ml = True
遍历    : [99, 2]


## 1.6 可调用对象：`__call__`

实现 `__call__` 后，实例可以像函数一样被调用——这是**装饰器类**和**回调对象**（观察者）的基础。

In [13]:
# ① 可调用对象（LLMOps 场景：预绑定参数的调用器，如固定 temperature 的补全函数）
class Adder:
    def __init__(self, n):
        self.n = n              # ② 构造时绑定偏移量

    def __call__(self, x):      # ③ 实例被调用时执行
        return x + self.n

add5 = Adder(5)
print("add5(10)     =", add5(10))
print("callable检验 =", callable(add5))     # ④ callable() 检查 __call__ 是否存在
print("普通对象    =", callable(object()))  # ⑤ 普通实例不可调用

add5(10)     = 15
callable检验 = True
普通对象    = False


## 1.7 上下文管理：`__enter__` / `__exit__`

`with` 语句保证 `__exit__` **无论如何（包括异常）都会执行**——这是资源释放的正确姿势（对比不可靠的 `__del__`）。

`__exit__` 返回 `True` 表示**吞掉异常**，返回 `False` 表示异常继续向外抛。

In [14]:
# ① 上下文管理协议
class Resource:
    def __enter__(self):
        print("1. __enter__  -> 获取资源")   # ② 进入 with 块，返回值绑定给 as 后的变量
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("3. __exit__   -> 释放资源")   # ③ 离开 with 块（正常或异常）必然执行
        return False                          # ④ False = 不吞异常，向上传播

with Resource() as r:
    print("2. with 块内 -> 使用资源")

# ⑤ 异常路径验证：即使 with 块内抛异常，__exit__ 依然执行
class SafeExit(Resource):
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("3. __exit__   -> 释放资源（异常时也执行，捕获到:",
              exc_type.__name__ if exc_type else None, "）")
        return True                           # ⑥ True = 吞掉异常，外面看不到

with SafeExit():
    raise ValueError("with 块内出错了")
print("异常被 __exit__ 吞掉，程序继续执行")

1. __enter__  -> 获取资源
2. with 块内 -> 使用资源
3. __exit__   -> 释放资源
1. __enter__  -> 获取资源
3. __exit__   -> 释放资源（异常时也执行，捕获到: ValueError ）
异常被 __exit__ 吞掉，程序继续执行


## 1.8 属性访问三剑客：`__getattr__` / `__setattr__` / `__delattr__`

| 方法 | 触发时机 |
|---|---|
| `__getattr__` | **常规查找失败后**才调用（兜底） |
| `__getattribute__` | **所有**属性读取都先调用（极易死循环） |
| `__setattr__` | 所有属性赋值 |
| `__delattr__` | 所有属性删除 |

In [15]:
# ① 属性访问钩子（执行顺序验证）
class Demo:
    def __init__(self):
        self.x = 1                             # ② 这一步就会触发 __setattr__

    def __getattr__(self, name):               # ③ 只有【找不到】属性时才被调用
        print(f"   __getattr__ 兜底: {name}")
        return None

    def __setattr__(self, name, value):        # ④ 所有赋值都会经过这里
        print(f"   __setattr__: {name} = {value}")
        super().__setattr__(name, value)       # ⑤ 必须委托给 object，否则真的死循环

    def __delattr__(self, name):
        print(f"   __delattr__: {name}")
        super().__delattr__(name)

print("读 d.x（存在）→", end=" ")
d = Demo.__new__(Demo)                         # ⑥ 绕过 __init__，避免干扰演示输出
object.__setattr__(d, "x", 1)
print(d.x)                                     # ⑦ 存在的属性：不触发 __getattr__
print("读 d.y（不存在）→")
print("   结果:", d.y)                          # ⑧ 不存在：触发 __getattr__
print("写 d.z:")
d.z = 2
print("删 d.z:")
del d.z

读 d.x（存在）→ 1
读 d.y（不存在）→
   __getattr__ 兜底: y
   结果: None
写 d.z:
   __setattr__: z = 2
删 d.z:
   __delattr__: z


### 反例：`__getattribute__` 的经典死循环

`__getattribute__` 拦截**一切**属性读取——包括你在它内部用到的 `self.__dict__`，于是自我调用直到 `RecursionError`。

In [16]:
# ① 错误写法：在 __getattribute__ 内部访问 self 的属性
class Bad:
    def __getattribute__(self, name):
        return self.__dict__[name]     # ② self.__dict__ 又触发 __getattribute__ → 无限递归

try:
    Bad().x
except RecursionError:
    print("错误写法 → RecursionError（无限递归）")

# ③ 正确写法：内部一律用 object.__getattribute__ / super()
class Good:
    def __getattribute__(self, name):
        print(f"   访问属性: {name}")
        return super().__getattribute__(name)   # ④ 委托给 object，绕开自身钩子

    def __init__(self):
        self.x = 1

g = Good()
print("正确写法 → g.x =", g.x)

错误写法 → RecursionError（无限递归）
   访问属性: x
正确写法 → g.x = 1


## 1.9 其他常用魔术方法速查

| 方法 | 作用 | 方法 | 作用 |
|---|---|---|---|
| `__bool__` | `bool(obj)` / 真值判断 | `__format__` | `format(obj, spec)` |
| `__int__` / `__float__` | 类型转换 | `__bytes__` | `bytes(obj)` |
| `__hash__` | `hash(obj)` | `__enter__` / `__exit__` | `with` |
| `__await__` | `await` | `__aiter__` / `__anext__` | `async for` |

In [17]:
# ① 真值与类型转换魔术方法（LLMOps 场景：对象为空时在 if 中自然判 False）
class Batch:
    def __init__(self, items):
        self.items = items

    def __bool__(self):                 # ② if batch: / bool(batch)
        return len(self.items) > 0

    def __len__(self):                  # ③ 未定义 __bool__ 时 Python 会退而求 __len__
        return len(self.items)

    def __format__(self, spec):         # ④ format(obj, spec)
        if spec == "count":
            return f"批量大小={len(self.items)}"
        return str(self.items)

empty, full = Batch([]), Batch(["a", "b"])
print("bool(empty) =", bool(empty), "| bool(full) =", bool(full))
print("if 判断     :", "空批次，跳过" if not empty else "处理批次")
print("format     :", format(full, "count"))

bool(empty) = False | bool(full) = True
if 判断     : 空批次，跳过
format     : 批量大小=2


# 第 2 章 属性管理

**本章解决什么问题**：不改变调用方式（`obj.attr`）的前提下，给属性加上校验、计算、缓存逻辑。

| 工具 | 定位 | 适用 |
|---|---|---|
| `property` | 已知属性的精确控制（读/写/删） | 校验、计算属性、只读 |
| `cached_property` | 计算一次后缓存到实例字典 | 昂贵计算 |
| `__getattr__` | 动态处理**未知**属性 | 代理、动态属性名 |

In [18]:
# ① property：带校验的读写属性（LLMOps 场景：模型参数 temperature 校验）
class Circle:
    def __init__(self, radius):
        self._radius = radius          # ② 注意：这里走 setter（先触发一次校验）

    @property                          # ③ getter：c.radius 调用
    def radius(self):
        return self._radius

    @radius.setter                     # ④ setter：c.radius = v 调用，可校验
    def radius(self, value):
        if value < 0:
            raise ValueError("半径不能为负")
        self._radius = value

    @property                          # ⑤ 计算属性：只读，每次访问现算
    def area(self):
        return 3.14159 * self._radius ** 2

c = Circle(5)
print("c.radius =", c.radius)
print("c.area   =", c.area)
c.radius = 10                          # ⑥ 走 setter
print("修改后 c.area =", c.area)
try:
    c.radius = -1                      # ⑦ 非法赋值
except ValueError as e:
    print("校验生效 →", type(e).__name__, ":", e)

c.radius = 5
c.area   = 78.53975
修改后 c.area = 314.159
校验生效 → ValueError : 半径不能为负


In [19]:
# ① 只读属性：只定义 getter，不定义 setter
class User:
    def __init__(self, name):
        self._name = name              # ② 内部字段带下划线（约定：外部不要直接碰）

    @property
    def name(self):                    # ③ 只有 getter → 对外只读
        return self._name

u = User("Alice")
print("只读访问:", u.name)
try:
    u.name = "Bob"                     # ④ 尝试写入
except AttributeError as e:
    print("写入被拒 →", type(e).__name__, ":", e)

只读访问: Alice
写入被拒 → AttributeError : property 'name' of 'User' object has no setter


In [20]:
# ① cached_property：只算一次（LLMOps 场景：词表/编码器只加载一次）
from functools import cached_property

class Data:
    def __init__(self, values):
        self.values = values

    @cached_property                   # ② 首次访问计算，结果写入实例 __dict__（在 __dict__ 中遮蔽同名）
    def total(self):
        print("   [计算中] 首次访问才执行")
        return sum(self.values)

d = Data([1, 2, 3])
print("第一次:", d.total)             # ③ 触发计算
print("第二次:", d.total)             # ④ 直接读缓存，不再打印「计算中」
print("缓存位置:", "total" in d.__dict__)   # ⑤ 验证：结果确实缓存在实例字典

   [计算中] 首次访问才执行
第一次: 6
第二次: 6
缓存位置: True


### `property` vs `__getattr__` 的选择

| 维度 | `property` | `__getattr__` |
|---|---|---|
| 属性是否已知 | 已知、数量有限 | 未知、动态生成 |
| 触发时机 | 每次访问都执行 getter | 仅常规查找**失败**后 |
| 典型场景 | 校验、计算属性 | 代理转发（如 `obj.data.xxx` → 远程字段） |

In [21]:
# ① 对比验证：property 每次都执行；__getattr__ 只在找不到时兜底
class Demo:
    def __init__(self):
        self._count = 0

    @property
    def prop(self):                    # ② 已知属性：每次访问都执行（计数可见）
        self._count += 1
        return f"property 第 {self._count} 次执行"

    def __getattr__(self, name):       # ③ 未知属性兜底：只有 self.__dict__ 里没有时才调用
        return f"__getattr__ 兜底了未知属性: {name}"

d = Demo()
print(d.prop, "|", d.prop)             # ④ property：两次访问 → 计数到 2
print(d.anything)                      # ⑤ 未定义属性 → __getattr__
print(d.prop)                          # ⑥ 已定义属性永远不会走 __getattr__

property 第 1 次执行 | property 第 2 次执行
__getattr__ 兜底了未知属性: anything
property 第 3 次执行


# 第 3 章 描述符（Descriptor）

**本章解决什么问题**：`property` 只能绑定一个属性；描述符把「属性逻辑」做成**可复用的类**，一处实现，处处声明——ORM 字段、参数校验的底层原理。

描述符 = 实现了 `__get__` / `__set__` / `__delete__` 中至少一个的**类属性对象**。`property` 本质上就是一个数据描述符。

| 类型 | 实现方法 | 优先级 |
|---|---|---|
| 数据描述符（data descriptor） | `__get__` + `__set__`（或 `__delete__`） | 高于实例字典 |
| 非数据描述符（non-data descriptor） | 仅 `__get__` | 低于实例字典 |

**属性查找优先级（背诵级结论）**：数据描述符 → 实例字典 → 非数据描述符 → 类属性 → `__getattr__`

## 3.1 数据描述符：类型校验

In [22]:
# ① 数据描述符：可复用的类型校验（LLMOps 场景：请求参数必须是整数）
class Integer:
    def __set_name__(self, owner, name):       # ② 类体执行时自动回调，告知「我绑定到了哪个属性名」
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:                        # ③ 通过类访问（Point.x）时返回描述符自身
            return self
        return obj.__dict__.get(self.name)     # ④ 值实际存在实例字典中，描述符只是守门人

    def __set__(self, obj, value):             # ⑤ 赋值时校验：数据描述符因此能拦截实例字典
        if not isinstance(value, int):
            raise TypeError(f"{self.name} 必须是整数")
        obj.__dict__[self.name] = value

class Point:
    x = Integer()                              # ⑥ 一行声明，逻辑完全复用
    y = Integer()

    def __init__(self, x, y):
        self.x = x                             # ⑦ 赋值 → 触发 Integer.__set__ 校验
        self.y = y

p = Point(1, 2)
print("p.x =", p.x, "| p.y =", p.y)
try:
    p.x = "a"                                  # ⑧ 非法类型
except TypeError as e:
    print("校验生效 →", type(e).__name__, ":", e)

p.x = 1 | p.y = 2
校验生效 → TypeError : x 必须是整数


## 3.2 非数据描述符：手写 `cached_property`

只实现 `__get__` 的描述符优先级**低于实例字典**——所以首次计算后把结果塞进 `obj.__dict__`，下次访问就直接命中实例字典，不再进描述符。这正是 `cached_property` 的原理。

In [23]:
# ① 非数据描述符实现惰性计算
class Lazy:
    def __init__(self, func):
        self.func = func

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = self.func(obj)                       # ② 首次访问：执行计算
        obj.__dict__[self.func.__name__] = value     # ③ 关键：写进实例字典（优先级高于非数据描述符）
        return value

class Data:
    def __init__(self, values):
        self.values = values

    @Lazy                                             # ④ 等价于 total = Lazy(计算函数)
    def total(self):
        print("   [计算中] 首次访问才执行")
        return sum(self.values)

d = Data([1, 2, 3])
print("第一次:", d.total)       # ⑤ 触发计算
print("第二次:", d.total)       # ⑥ 命中实例字典缓存，不再计算

   [计算中] 首次访问才执行
第一次: 6
第二次: 6


## 3.3 描述符优先级验证

用「同名实例字典项 vs 描述符」对撞，直接证明优先级结论。

In [24]:
# ① 优先级实验：数据描述符 vs 实例字典 vs 非数据描述符
class DataDesc:
    def __get__(self, obj, objtype=None):
        return "data desc"                 # ② 数据描述符：有 __set__
    def __set__(self, obj, value):
        pass                               # ③ 空 set 也算「数据描述符」

class NonDataDesc:
    def __get__(self, obj, objtype=None):
        return "non-data desc"             # ④ 非数据描述符：只有 __get__

class Demo:
    data = DataDesc()
    non_data = NonDataDesc()

d = Demo()
d.__dict__["data"] = "instance"            # ⑤ 直接往实例字典塞同名键，模拟「竞争」
d.__dict__["non_data"] = "instance"

print("d.data     =", d.data)              # ⑥ 数据描述符赢 → "data desc"
print("d.non_data =", d.non_data)          # ⑦ 实例字典赢 → "instance"
# 结论（已用输出证明）：数据描述符 > 实例字典 > 非数据描述符

d.data     = data desc
d.non_data = instance


## 3.4 描述符的典型使用场景

| 场景 | 做法 |
|---|---|
| ORM 字段 | `name = Field()` 声明式建模 |
| 类型校验 | `__set__` 中 `isinstance` 检查 |
| 延迟计算 | 非数据描述符 + 实例字典缓存 |
| 属性代理 | `__get__` 转发到内部对象 |

In [25]:
# ① 通用 Field 描述符：ORM 建模雏形（LLMOps 场景：模型元数据表）
class Field:
    def __set_name__(self, owner, name):
        self.name = name                    # ② 记住绑定名

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        obj.__dict__[self.name] = value     # ③ 存取都走实例字典，描述符只负责「挂载逻辑」

class Model:
    def __init__(self, **kwargs):           # ④ 通用构造：字段在类体声明，值在实例字典
        for k, v in kwargs.items():
            setattr(self, k, v)             # ⑤ setattr → 触发各字段的 __set__

class ModelMeta(Model):                     # ⑥ 一行一个字段，声明式建模
    id = Field()
    name = Field()
    provider = Field()

m = ModelMeta(id=1, name="gpt-4o", provider="openai")
print("字段值:", m.id, m.name, m.provider)
print("存储位置:", m.__dict__)

字段值: 1 gpt-4o openai
存储位置: {'id': 1, 'name': 'gpt-4o', 'provider': 'openai'}


# 第 4 章 继承与 MRO

**本章解决什么问题**：多继承时方法到底走哪条路径？`super()` 到底调用谁？如何让多个父类优雅协作？

MRO（Method Resolution Order，方法解析顺序）由 **C3 线性化算法** 计算，保证：
1. 子类优先于父类
2. 多个父类按**声明顺序**
3. 单调性（每个类在整个顺序中只出现一次且顺序一致）

In [26]:
# ① 菱形继承与 MRO
class A:
    def f(self): print("A")
class B(A):
    def f(self): print("B")
class C(A):
    def f(self): print("C")
class D(B, C):    # ② 菱形：D -> B -> C -> A（B、C 都继承 A）
    pass

print("MRO:", [c.__name__ for c in D.__mro__])   # ③ 查看线性化顺序
D().f()           # ④ 按 MRO 找到的第一个实现是 B

MRO: ['D', 'B', 'C', 'A', 'object']
B


## 4.1 `super()` 不是「父类」

**`super()` 按 MRO 查找「下一个」类**，与「直接父类」无关。协作式多继承里，同一个 `super().f()` 在不同类中指向完全不同的目标。

In [27]:
# ① 验证：super() 沿 MRO 链传递，而非固定调用父类
class A:
    def f(self): print("A")
class B(A):
    def f(self):
        print("B"); super().f()     # ② B 的 super → MRO 中 B 的下一个 = C（不是 A！）
class C(A):
    def f(self):
        print("C"); super().f()     # ③ C 的 super → A
class D(B, C):
    def f(self):
        print("D"); super().f()     # ④ D 的 super → B

D().f()                             # ⑤ 输出顺序 D B C A —— 一条链走到底，A 只执行一次
print("MRO:", [c.__name__ for c in D.__mro__])

D
B
C
A
MRO: ['D', 'B', 'C', 'A', 'object']


## 4.2 协作式多继承：`**kwargs` 贯穿链

每个类**只认领自己的参数**，其余参数用 `**kwargs` 继续下传，最终由 `object` 收口。这是 mixin 组合的标准姿势。

In [28]:
# ① 协作式多继承（LLMOps 场景：服务需要 log 与 path 两个配置维度）
class Base:
    def __init__(self, **kwargs):
        super().__init__(**kwargs)          # ② 链条终点：object.__init__() 不接受参数，收口

class Logger(Base):
    def __init__(self, log=None, **kwargs): # ③ 只认领 log，其余下传
        super().__init__(**kwargs)
        self.log = log

class Storage(Base):
    def __init__(self, path=None, **kwargs):# ④ 只认领 path，其余下传
        super().__init__(**kwargs)
        self.path = path

class App(Logger, Storage):                 # ⑤ 两个功能维度自由组合
    pass

app = App(log=True, path="/tmp/llmops")
print("app.log =", app.log, "| app.path =", app.path)
print("MRO:", [c.__name__ for c in App.__mro__])

app.log = True | app.path = /tmp/llmops
MRO: ['App', 'Logger', 'Storage', 'Base', 'object']


## 4.3 Mixin：只提供功能、不单独实例化的小类

约定：命名以 `Mixin` 结尾；放在继承列表**最左边**；`__init__` 用 `**kwargs` 协作。

In [29]:
# ① Mixin 组合（LLMOps 场景：模型对象需要 JSON 序列化与调试表示）
class JsonMixin:
    def to_json(self):                       # ② 提供能力一：序列化
        import json
        return json.dumps(self.__dict__, ensure_ascii=False)

class ReprMixin:
    def __repr__(self):                      # ③ 提供能力二：调试表示
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{self.__class__.__name__}({attrs})"

class ModelInfo(JsonMixin, ReprMixin):       # ④ 业务类只写业务字段
    def __init__(self, name, provider):
        self.name = name
        self.provider = provider

m = ModelInfo("gpt-4o", "openai")
print("print 走 ReprMixin :", m)
print("json 走 JsonMixin  :", m.to_json())

print 走 ReprMixin : ModelInfo(name='gpt-4o', provider='openai')
json 走 JsonMixin  : {"name": "gpt-4o", "provider": "openai"}


# 第 5 章 抽象基类（ABC）

**本章解决什么问题**：定义「接口契约」——子类必须实现哪些方法，否则连实例化都不允许。这是插件体系的根基。

- `ABC` + `@abstractmethod`：未实现全部抽象方法的子类无法实例化
- 虚拟子类 `register()`：不需要真实继承也能纳入 `issubclass` 体系（鸭子类型的正式化）

In [30]:
# ① 抽象基类：Provider 接口约束（LLMOps 场景）
from abc import ABC, abstractmethod

class Provider(ABC):
    @abstractmethod
    def invoke(self, prompt: str) -> str:    # ② 契约：每个 Provider 必须实现 invoke
        ...

class MockProvider(Provider):
    def invoke(self, prompt):                # ③ 实现契约
        return f"[mock] {prompt}"

try:
    Provider()                               # ④ 抽象类不能实例化
except TypeError as e:
    print("抽象类实例化 →", type(e).__name__, ":", str(e)[:60], "...")

p = MockProvider()
print("子类正常调用:", p.invoke("你好"))

抽象类实例化 → TypeError : Can't instantiate abstract class Provider without an impleme ...
子类正常调用: [mock] 你好


In [31]:
# ① 抽象属性：@property 与 @abstractmethod 叠加
from abc import ABC, abstractmethod

class Shape(ABC):
    @property
    @abstractmethod                    # ② 顺序：property 在外层（先装饰成 property，再标记抽象）
    def name(self): ...

class Circle(Shape):
    @property
    def name(self):                    # ③ 子类必须用同名 property 覆盖
        return "圆"

print("name =", Circle().name)
try:
    Shape()                            # ④ 未实现抽象属性同样不能实例化
except TypeError as e:
    print("未实现抽象属性 →", type(e).__name__)

name = 圆
未实现抽象属性 → TypeError


In [32]:
# ① 虚拟子类：register 让第三方类「挂靠」到 ABC 体系
from abc import ABC

class MyABC(ABC):
    def hello(self):
        return "来自 MyABC 的默认实现"

class MyClass:                          # ② 完全独立的类，没有继承 MyABC
    def hello(self):
        return "我是虚拟子类"

MyABC.register(MyClass)                 # ③ 注册为虚拟子类
print("issubclass(MyClass, MyABC) =", issubclass(MyClass, MyABC))  # ④ True
print("isinstance 实例检查       =", isinstance(MyClass(), MyABC))
print("但不会继承实现:", MyClass().hello())   # ⑤ register 只影响类型检查，不注入任何方法

issubclass(MyClass, MyABC) = True
isinstance 实例检查       = True
但不会继承实现: 我是虚拟子类


In [33]:
# ① 常用内置 ABC：面向协议编程（推荐用 collections.abc 而不是 collections）
from collections.abc import Iterable, Sequence, Mapping

print("list   是 Sequence :", isinstance([1, 2], Sequence))       # ② 有序、可索引、有长度
print("dict   是 Mapping  :", isinstance({"a": 1}, Mapping))      # ③ 键值映射协议
print("生成器 是 Iterable :", isinstance((x for x in range(3)), Iterable))  # ④ 可迭代但不可索引
print("生成器 是 Sequence :", isinstance((x for x in range(3)), Sequence))  # ⑤ False：协议不同

list   是 Sequence : True
dict   是 Mapping  : True
生成器 是 Iterable : True
生成器 是 Sequence : False


# 第 6 章 元类

**本章解决什么问题**：拦截「类的创建过程」本身，做注册、校验、改写——框架（Django ORM、SQLAlchemy、pydantic v1）的底层魔法。

元类 = 创建类的类。`class` 语句执行时，解释器调用 `metaclass(name, bases, namespace)` 生成类对象；默认元类是 `type`。

**优先级建议（背诵）**：`__init_subclass__` → 类装饰器 → 元类。元类威力最大、成本也最高（难调试、易过度设计）。

In [34]:
# ① type 三参数形式：动态创建类（元类的底层入口）
User = type("User", (), {"name": "Alice", "greet": lambda self: f"hi, {self.name}"})
u = User()
print("u.name =", u.name)
print("u.greet() =", u.greet())
print("type(User) =", type(User))          # ② 这个类确实由 type 创建

u.name = Alice
u.greet() = hi, Alice
type(User) = <class 'type'>


In [35]:
# ① 自定义元类：拦截类创建的两个钩子
class Meta(type):
    def __new__(mcls, name, bases, namespace):
        # ② __new__：类对象【创建之前】调用，可改写 namespace（增删属性）
        print(f"1. Meta.__new__  创建类: {name}")
        cls = super().__new__(mcls, name, bases, namespace)
        return cls

    def __init__(cls, name, bases, namespace):
        # ③ __init__：类对象【创建之后】调用，可做注册等收尾
        super().__init__(name, bases, namespace)
        print(f"2. Meta.__init__ 初始化类: {name}")

class User(metaclass=Meta):    # ④ 声明元类；下面的输出发生在【类定义时刻】，而非实例化时刻
    pass

print("3. 类定义完成后才执行到这里")

1. Meta.__new__  创建类: User
2. Meta.__init__ 初始化类: User
3. 类定义完成后才执行到这里


In [36]:
# ① 元类实战：注册模式（框架插件的底层原理）
class RegistryMeta(type):
    registry = {}                                   # ② 元类身上的全局注册表

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        if bases:                                   # ③ 跳过基类自身，只注册真实子类
            mcls.registry[name] = cls               # ④ 类定义即注册，无需手动调用
        return cls

class Base(metaclass=RegistryMeta):
    pass

class User(Base):    # ⑤ 定义即注册
    pass

class Order(Base):
    pass

print("注册表:", list(RegistryMeta.registry))

注册表: ['User', 'Order']


## 6.1 `__init_subclass__`：元类的主要替代品

95% 的「类创建时拦截」需求，用类方法 `__init_subclass__` 就够了——不需要自定义元类、不引发元类冲突。

In [37]:
# ① __init_subclass__：子类定义时自动回调（推荐替代元类注册）
class Base:
    registry = {}

    def __init_subclass__(cls, **kwargs):           # ② 每当有子类被定义，自动调用
        super().__init_subclass__(**kwargs)         # ③ 保持链式协作
        Base.registry[cls.__name__] = cls           # ④ 同样实现「定义即注册」

class User(Base):  pass
class Order(Base): pass

print("注册表:", list(Base.registry))
# 对比：功能与 RegistryMeta 完全等价，但没有引入自定义元类

注册表: ['User', 'Order']


In [38]:
# ① __set_name__：描述符知道自己挂在哪个属性上（第 3 章 Integer 的回马枪）
class Field:
    def __set_name__(self, owner, name):            # ② 类体执行时自动回调
        self.name = name
        print(f"   {owner.__name__}.{name} 完成绑定")

class User:
    name = Field()                                  # ③ 类定义时即触发绑定输出
    age = Field()

print("绑定完成，Field 自己知道属性名:", User.name.name, User.age.name)
# 价值：描述符不再需要显式传属性名，声明处零冗余

   User.name 完成绑定
   User.age 完成绑定
绑定完成，Field 自己知道属性名: name age


### 元类三选一：决策表

| 方案 | 能力 | 成本 | 优先级 |
|---|---|---|---|
| `__init_subclass__` | 子类定义时回调（注册、简单校验） | 极低 | **首选** |
| 类装饰器 | 创建后改写类（注入方法、包装） | 低 | 其次 |
| 自定义元类 | 创建**前**改写命名空间、控制类的一切 | 高（难调试、元类冲突） | 最后 |

In [39]:
# ① 类装饰器：另一个轻量替代（给类注入方法）
def with_counter(cls):
    cls.created = 0                                # ② 类属性：实例计数
    original_init = cls.__init__

    def counting_init(self, *args, **kwargs):
        cls.created += 1                           # ③ 每次实例化 +1
        original_init(self, *args, **kwargs)
    cls.__init__ = counting_init
    return cls

@with_counter                                      # ④ 装饰类而非函数
class Task:
    def __init__(self, name):
        self.name = name

Task("a"); Task("b")
print("实例计数:", Task.created)

实例计数: 2


# 第 7 章 数据类与 `__slots__`

**本章解决什么问题**：数据载体类（配置、请求、响应）的样板代码消除，以及实例内存优化。

In [40]:
# ① dataclass 基本用法（LLMOps 场景：模型调用请求体）
from dataclasses import dataclass, field

@dataclass
class CompletionRequest:
    model: str                                    # ② 无默认值的字段必须在前
    prompt: str
    temperature: float = 0.7                      # ③ 有默认值
    tags: list[str] = field(default_factory=list) # ④ 可变默认值必须用 default_factory！

req = CompletionRequest("gpt-4o", "你好")
print("repr 自动生成:", req)
print("eq  自动生成:", req == CompletionRequest("gpt-4o", "你好"))
print("默认值互不影响:", req.tags is not CompletionRequest("a", "b").tags)  # ⑤ 两个独立 list

repr 自动生成: CompletionRequest(model='gpt-4o', prompt='你好', temperature=0.7, tags=[])
eq  自动生成: True
默认值互不影响: True


In [41]:
# ① dataclass 常用参数一览
from dataclasses import dataclass

@dataclass(order=True, frozen=True, slots=True)   # ② 三连开
class Point:
    x: float
    y: float

p1, p2 = Point(1, 2), Point(1, 3)
print("order=True  → 可比较/排序:", sorted([p2, p1]) == [p1, p2])
print("frozen=True → 可哈希:", {p1: "起点"}[Point(1, 2)])
try:
    p1.x = 99                                     # ③ 不可变：赋值报错
except Exception as e:
    print("frozen=True → 不可变:", type(e).__name__)
print("slots=True  → 无实例字典:", not hasattr(p1, "__dict__"))   # ④ 省内存的根源

order=True  → 可比较/排序: True
frozen=True → 可哈希: 起点
frozen=True → 不可变: FrozenInstanceError
slots=True  → 无实例字典: True


In [42]:
# ① dataclass 支持继承（字段顺序：父类字段在前，带默认值的注意顺序）
from dataclasses import dataclass

@dataclass
class BaseRequest:
    model: str                                    # ② 父类：无默认值字段

@dataclass
class ChatRequest(BaseRequest):
    prompt: str = ""                              # ③ 子类字段跟在后面
    stream: bool = False

r = ChatRequest(model="gpt-4o", prompt="你好")
print("继承字段:", r)
print("字段顺序:", [f.name for f in __import__("dataclasses").fields(r)])

继承字段: ChatRequest(model='gpt-4o', prompt='你好', stream=False)
字段顺序: ['model', 'prompt', 'stream']


In [43]:
# ① 与普通类对比：dataclass 消灭的样板代码
from dataclasses import dataclass

# ② 普通类：4 个方法全是样板
class Point1:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"Point1({self.x}, {self.y})"
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

# ③ dataclass：6 行变 3 行，且额外免费获得 __ne__ 等
@dataclass
class Point2:
    x: float
    y: float

print("功能等价:", Point1(1, 2) == Point1(1, 2), "|", Point2(1, 2) == Point2(1, 2))
print("表示等价:", repr(Point1(1, 2)), "==", repr(Point2(1, 2)))

功能等价: True | True
表示等价: Point1(1, 2) == Point2(x=1, y=2)


## 7.1 `NamedTuple`：轻量不可变数据

| 维度 | `NamedTuple` | `@dataclass(frozen=True)` |
|---|---|---|
| 本质 | tuple 子类 | 普通类 |
| 内存 | 更小 | 较小（配 slots 更小） |
| 解包 | 支持 `x, y = p` | 不支持 |
| 方法/默认值 | 支持（受限） | 完整支持 |

In [44]:
# ① NamedTuple（LLMOps 场景：轻量的使用统计返回值）
from typing import NamedTuple

class Usage(NamedTuple):
    prompt_tokens: int
    completion_tokens: int
    model: str = "gpt-4o"          # ② 支持默认值

u = Usage(100, 50)
print("字段访问:", u.prompt_tokens, u.completion_tokens)
print("元组解包:", f"输入 {u[0]} + 输出 {u[1]} = {sum(u[:2])} tokens")
print("不可变 & 可哈希:", {u: "first"}[Usage(100, 50)])

字段访问: 100 50
元组解包: 输入 100 + 输出 50 = 150 tokens
不可变 & 可哈希: first


## 7.2 `__slots__`：限制属性 + 节省内存

原理：告诉解释器「实例只有这些属性」，于是**不再为每个实例创建 `__dict__`」，属性直接存进固定的槽位。

In [45]:
# ① __slots__ 基本行为
class Point:
    __slots__ = ("x", "y")         # ② 白名单：只允许 x、y

    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(1, 2)
print("白名单内正常:", p.x, p.y)
try:
    p.z = 3                        # ③ 白名单外
except AttributeError as e:
    print("白名单外 →", type(e).__name__, ":", e)
print("无实例字典:", not hasattr(p, "__dict__"))

白名单内正常: 1 2
白名单外 → AttributeError : 'Point' object has no attribute 'z' and no __dict__ for setting new attributes
无实例字典: True


In [46]:
# ① 内存对比（量化收益）
import sys

class A:                            # ② 普通类：每个实例背一个 __dict__
    def __init__(self, x):
        self.x = x

class B:                            # ③ slots 类：无 __dict__
    __slots__ = ("x",)
    def __init__(self, x):
        self.x = x

a, b = A(1), B(1)
print("普通类实例字典大小:", sys.getsizeof(a.__dict__), "bytes")   # ④ 典型 ~100+ bytes
print("slots 类有无字典  :", hasattr(b, "__dict__"))               # ⑤ False
print("实例本体大小对比   :", sys.getsizeof(a), "vs", sys.getsizeof(b))
# 大量小对象（如 token、事件）场景下，slots 能省 30%+ 内存

普通类实例字典大小: 296 bytes
slots 类有无字典  : False
实例本体大小对比   : 48 vs 40


In [47]:
# ① __slots__ 与继承的坑：子类不声明 __slots__ 会重新获得 __dict__
class Base:
    __slots__ = ("x",)

class Good(Base):
    __slots__ = ("y",)              # ② 子类也声明：仍然没有 __dict__

class Bad(Base):
    pass                            # ③ 子类不声明：__dict__ 回来了，省内存失效

g, bad = Good(), Bad()
g.x, g.y = 1, 2
bad.x, bad.anything = 1, "ok"       # ④ Bad 想加什么属性都行
print("Good 无字典:", not hasattr(g, "__dict__"))
print("Bad  有字典:", hasattr(bad, "__dict__"), "->", bad.__dict__)

Good 无字典: True
Bad  有字典: True -> {'anything': 'ok'}


# 第 8 章 工程高频模式精选（实用示例集）

**本章解决什么问题**：把前面的机制落到 5 个日常开发最高频的模式。每个模式给出适用场景与注意点。

| # | 模式 | 核心机制 | 适用场景 | 注意点 |
|---|---|---|---|---|
| 1 | 单例配置 | `__new__` | 全局配置、连接池 | 子类 `__init__` 会重复执行 |
| 2 | 类型校验描述符 | 数据描述符 + `__set_name__` | 请求参数/配置校验 | 值存实例字典，不是描述符里 |
| 3 | 插件注册 | `__init_subclass__` | Provider/Handler 扩展 | 用它别用元类 |
| 4 | 缓存计算 | `cached_property` / 非数据描述符 | 词表、编码器加载 | 实例可变时注意缓存失效 |
| 5 | 上下文计时器 | `__enter__`/`__exit__` | 耗时观测、资源管理 | 返回 False 别吞异常 |

In [48]:
# ① 模式 1：单例配置中心（LLMOps 场景：全局 Settings）
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

class Config(Singleton):
    def __init__(self):
        if not hasattr(self, "_initialized"):     # ② 防止 __init__ 重复执行的守卫
            self.debug = False
            self._initialized = True               # ③ 标记：后续 Singleton() 再调用也不会重置

a = Config()
a.debug = True
b = Config()           # ④ 第二次调用：__new__ 返回同一实例，__init__ 因守卫跳过
print("a is b =", a is b, "| b.debug =", b.debug)  # ⑤ debug 仍是 True，未被重置

a is b = True | b.debug = True


> **修正说明 2（源文档勘误）**：源文档 9.2 的 `Config(Singleton)` 示例中，`__init__` 无守卫——第二次 `Config()` 会把已修改的属性**重置回默认值**（`__new__` 复用实例，但 `__init__` 每次都跑）。上面代码用 `_initialized` 守卫修复，输出证明 `debug=True` 得以保留。

In [49]:
# ① 模式 2：类型校验描述符族（描述符继承复用）
class Typed:
    expected_type = object                      # ② 基类定义协议骨架

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} 必须是 {self.expected_type.__name__}")
        obj.__dict__[self.name] = value

class IntegerField(Typed):
    expected_type = int                          # ③ 子类只改类型 → 一行一个字段类型

class StringField(Typed):
    expected_type = str

class User:
    age = IntegerField()                         # ④ 声明式校验（LLMOps 场景：请求体字段）
    name = StringField()

    def __init__(self, age, name):
        self.age = age
        self.name = name

u = User(30, "Alice")
print("合法赋值:", u.age, u.name)
try:
    u.age = "x"
except TypeError as e:
    print("非法赋值 →", type(e).__name__, ":", e)

合法赋值: 30 Alice
非法赋值 → TypeError : age 必须是 int


In [50]:
# ① 模式 3：插件注册（LLMOps 场景：多模型 Provider 接入）
class Plugin:
    registry = {}

    def __init_subclass__(cls, **kwargs):        # ② 子类定义即注册，零手动调用
        super().__init_subclass__(**kwargs)
        Plugin.registry[cls.__name__] = cls

class PDFPlugin(Plugin):   pass                  # ③ 新插件 = 定义一个新类，注册自动完成
class ImagePlugin(Plugin): pass

print("已注册插件:", list(Plugin.registry))
new_cls = Plugin.registry["PDFPlugin"]           # ④ 按名取类、动态实例化
print("动态实例化:", type(new_cls()).__name__)

已注册插件: ['PDFPlugin', 'ImagePlugin']
动态实例化: PDFPlugin


In [51]:
# ① 模式 4：缓存计算（LLMOps 场景：tokenizer 只加载一次）
from functools import cached_property

class Tokenizer:
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size

    @cached_property
    def vocab(self):                             # ② 昂贵资源：首次访问才构建
        print("   [加载词表] 只应出现一次")
        return {f"tok{i}": i for i in range(self.vocab_size)}

tok = Tokenizer(100)
_ = tok.vocab                                     # ③ 第一次：触发加载
_ = tok.vocab                                     # ④ 第二次：直接读缓存
print("缓存验证: vocab 在实例字典 =", "vocab" in tok.__dict__)

   [加载词表] 只应出现一次
缓存验证: vocab 在实例字典 = True


In [52]:
# ① 模式 5：上下文计时器（LLMOps 场景：观测每次调用耗时）
import time

class Timer:
    def __enter__(self):
        self._start = time.perf_counter()        # ② 记录起点
        return self                              # ③ 返回自身，让外部拿到 elapsed

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start   # ④ 无论成败都记录
        return False                             # ⑤ 不吞异常

with Timer() as t:
    sum(range(1_000_000))                        # ⑥ 模拟一次推理耗时
print("计时完成，elapsed >= 0:", t.elapsed >= 0)  # ⑦ 用布尔断言代替原始耗时，保证输出确定

计时完成，elapsed >= 0: True


# 第 9 章 内置与生态对照

**本章解决什么问题**：同一个需求（数据建模 / 校验），标准库和生态里各有什么武器，怎么选。

| 工具 | 来源 | 可变性 | 校验能力 | 生成能力 | 典型场景 |
|---|---|---|---|---|---|
| `property` | 内置 | 可变 | 自定义逻辑 | — | 单属性校验、计算属性 |
| `functools.cached_property` | 标准库 | 可变（可手动失效） | — | — | 昂贵计算缓存 |
| `@dataclass` | 标准库 | 默认可变（`frozen` 可不可变） | 仅类型注解，**不强制** | `__init__`/`__repr__`/`__eq__`/`order` | 数据载体、配置对象 |
| `typing.NamedTuple` | 标准库 | 不可变 | 无 | 同上（受限） | 轻量返回值、坐标 |
| `enum.Enum` | 标准库 | 不可变 | 成员封闭 | — | 状态机、常量集合 |
| `attrs` | 第三方 | 可配置 | validators / converters | 最全 | 复杂领域模型 |
| `pydantic` | 第三方 | 可配置 | **运行时强制校验** | 全套 + JSON Schema | API 输入输出、LLM 结构化输出 |
| `abc.ABC` | 标准库 | — | 抽象方法约束 | — | 接口契约、插件体系 |

**选型口诀**：内部数据用 `dataclass`；对外边界用 `pydantic`；接口约束用 `ABC`；单属性逻辑用 `property`。

In [53]:
# ① 对照验证：dataclass 的注解【不】做运行时校验（与 pydantic 的本质区别）
from dataclasses import dataclass

@dataclass
class Request:
    temperature: float      # ② 仅仅是「注解」，赋什么都不拦
    model: str

r = Request(temperature="滚烫的", model=123)   # ③ 类型完全错误也能构造成功
print("dataclass 不校验:", r)
print("注解仍可读取:", Request.__annotations__["temperature"])

# ④ 真需要强制校验时的两个标准库方案：描述符（见模式2）或手动 property（见第2章）
# 生态方案：pip install pydantic 后 BaseModel 会直接抛 ValidationError

dataclass 不校验: Request(temperature='滚烫的', model=123)
注解仍可读取: <class 'float'>


# 第 10 章 综合实战：LLMOps 调用网关

**本章解决什么问题**：把 **描述符校验 + ABC 接口 + `__init_subclass__` 注册 + 协作式 Mixin + 上下文计时 + 观察者 + `cached_property` + MRO** 组合成一个微型但完整的 LLM 调用网关。

**业务场景（假设已与需求方对齐）**：LLMOps API 服务需要——
1. 调用参数（model / temperature / max_tokens）在赋值瞬间完成类型与范围校验
2. 新增模型 Provider 时**零改动**主流程（插件化）
3. 统一观测：每次调用的耗时与事件（观察者解耦）
4. 热点 Prompt 结果缓存

**设计取舍**：
- 校验用**描述符**而非 pydantic：避免外部依赖，且要支撑「范围校验」这种带参数的逻辑（描述符可继承）
- Provider 用 **ABC + `__init_subclass__`** 而非元类：能力够用、无元类冲突风险
- 观测用 **Mixin + 观察者**而非硬编码日志：新增订阅方（告警、审计）不改核心类
- 配置对象用 **frozen dataclass**：防篡改、可哈希、可作缓存键

In [54]:
# ① 组件一：带范围校验的参数描述符（第 3 章描述符 + 第 6 章 __set_name__ + 继承复用）
class Typed:
    expected_type = object

    def __set_name__(self, owner, name):
        self.name = name                       # ② 自动感知属性名

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} 必须是 {self.expected_type.__name__}")
        obj.__dict__[self.name] = value

class StrField(Typed):
    expected_type = str

class FloatField(Typed):
    expected_type = float

class IntRangeField(Typed):
    expected_type = int
    def __init__(self, lo, hi):                # ③ 带参数的描述符：比 pydantic 更灵活的自定义空间
        self.lo, self.hi = lo, hi
    def __set__(self, obj, value):
        super().__set__(obj, value)            # ④ 复用父类类型校验（super 走 MRO）
        if not (self.lo <= value <= self.hi):
            raise ValueError(f"{self.name} 需在 [{self.lo}, {self.hi}] 内")

class LLMParams:
    model = StrField()                          # ⑤ 声明式建模：一行一个受控参数
    temperature = FloatField()
    max_tokens = IntRangeField(1, 32768)

    def __init__(self, model, temperature, max_tokens):
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens

p = LLMParams("gpt-4o", 0.7, 4096)
print("合法参数:", p.model, p.temperature, p.max_tokens)
try:
    p.max_tokens = 99999999                     # ⑥ 范围拦截
except ValueError as e:
    print("范围校验 →", type(e).__name__, ":", e)
try:
    p.temperature = "hot"                       # ⑦ 类型拦截
except TypeError as e:
    print("类型校验 →", type(e).__name__, ":", e)

合法参数: gpt-4o 0.7 4096
范围校验 → ValueError : max_tokens 需在 [1, 32768] 内
类型校验 → TypeError : temperature 必须是 float


In [55]:
# ① 组件二：Provider 插件体系（第 5 章 ABC + 第 6 章 __init_subclass__）
from abc import ABC, abstractmethod

class Provider(ABC):
    registry = {}                                           # ② 全局注册表

    def __init_subclass__(cls, **kwargs):                   # ③ 定义即注册
        super().__init_subclass__(**kwargs)
        Provider.registry[cls.__name__] = cls

    @abstractmethod
    def invoke(self, prompt: str, **params) -> str:         # ④ 契约：必须实现 invoke
        ...

class MockProvider(Provider):
    # 本地 Mock：单测/开发环境用，不花钱
    def invoke(self, prompt, **params):
        return f"[mock:{params.get('model', '?')}] {prompt}"

class EchoProvider(Provider):
    # 回声 Provider：验证链路的探针
    def invoke(self, prompt, **params):
        return f"[echo] {prompt}"

try:
    Provider()                                # ⑤ 抽象类拦截
except TypeError:
    print("抽象类不可实例化 ✓")
print("已注册 Provider:", list(Provider.registry))
provider = Provider.registry["MockProvider"]()  # ⑥ 按名动态构建，主流程零 if-else
print("动态调用:", provider.invoke("你好", model="gpt-4o"))

抽象类不可实例化 ✓
已注册 Provider: ['MockProvider', 'EchoProvider']
动态调用: [mock:gpt-4o] 你好


In [56]:
# ① 组件三：可观测网关（Mixin 协作 + 观察者 + Timer + cached_property，多知识点合体）
import time
from functools import cached_property

class Timer:                                       # ② 第 1 章上下文管理：观测耗时
    def __enter__(self):
        self._start = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start
        return False

class EventMixin:                                  # ③ 观察者模式：解耦事件生产与消费
    def __init__(self, **kwargs):
        super().__init__(**kwargs)                 # ④ 协作式：参数继续下传（第 4 章）
        self._observers = []
    def subscribe(self, fn):
        self._observers.append(fn)
    def emit(self, event):
        for obs in self._observers:                # ⑤ 所有订阅者收到通知
            obs(event)

class CacheMixin:                                  # ⑥ 缓存代理模式
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._cache = {}
    def cached_call(self, key, factory):
        if key not in self._cache:
            self.emit(f"cache-miss: {key}")        # ⑦ 未命中才真正调用（并广播事件）
            self._cache[key] = factory()
        return self._cache[key]

class LLMGateway(EventMixin, CacheMixin):          # ⑧ 两个维度自由组合
    def __init__(self, provider, **kwargs):
        super().__init__(**kwargs)
        self.provider = provider

    @cached_property
    def capabilities(self):                        # ⑨ 能力清单只计算一次
        print("   [能力探测] 只应出现一次")
        return {"streaming": True, "models": ["gpt-4o", "mock-1"]}

    def complete(self, prompt, **params):
        with Timer() as t:                         # ⑩ 计时包住真实调用
            result = self.provider.invoke(prompt, **params)
        self.emit(f"invoke-done: model={params.get('model', '?')}, "
                  f"耗时已记录={t.elapsed >= 0}")
        return result

print("MRO:", [c.__name__ for c in LLMGateway.__mro__])   # ⑪ 验证协作链完整

MRO: ['LLMGateway', 'EventMixin', 'CacheMixin', 'object']


In [57]:
# ① 组装运行：全链路演示
gw = LLMGateway(provider=Provider.registry["MockProvider"]())

# ② 订阅事件：日志器（可再加告警、审计，均不改 LLMGateway —— 观察者的价值）
events = []
gw.subscribe(events.append)

# ③ 首次调用：真实执行 + cache-miss 事件
r1 = gw.cached_call("prompt:你好", lambda: gw.complete("你好", model="gpt-4o"))
print("第一次结果:", r1)
# ④ 第二次调用：命中缓存（不会再有 invoke-done 事件）
r2 = gw.cached_call("prompt:你好", lambda: gw.complete("你好", model="gpt-4o"))
print("第二次结果:", r2, "（来自缓存）")

# ⑤ 能力清单：cached_property 只探测一次
_ = gw.capabilities
_ = gw.capabilities

# ⑥ 事件流回顾：完整可审计
print("事件流:")
for ev in events:
    print("   -", ev)

# ⑦ 参数校验组件独立可用（组件一的复用性证明）
params = LLMParams("mock-1", 0.5, 1024)
print("受控参数:", params.model, params.temperature, params.max_tokens)

第一次结果: [mock:gpt-4o] 你好
第二次结果: [mock:gpt-4o] 你好 （来自缓存）
   [能力探测] 只应出现一次
事件流:
   - cache-miss: prompt:你好
   - invoke-done: model=gpt-4o, 耗时已记录=True
受控参数: mock-1 0.5 1024


# 第 11 章 常见陷阱（反例集）

**本章解决什么问题**：每个陷阱按「陷阱 / 症状 / 解法」给出，且全部配有**可运行的复现代码**——错误写法真实触发（`try/except` 捕获展示），正确写法对比输出。

| # | 陷阱 | 症状 | 解法 |
|---|---|---|---|
| 1 | 类变量放可变对象 | 所有实例共享同一 list | 在 `__init__` 里创建实例属性 |
| 2 | 只定义 `__eq__` | 对象不可哈希，不能进 dict/set | 同步定义 `__hash__` |
| 3 | 以为 `super()` 是父类 | 多继承下调用顺序反直觉 | 按 MRO 理解 super |
| 4 | 元类冲突 | 多继承时报 metaclass TypeError | 元类间保持子类关系 |
| 5 | `__del__` 释放资源 | 时机不确定甚至不调用 | 用上下文管理器 |
| 6 | 子类覆盖父类 property | 静默失效，难排查 | 子类用同名 property 覆盖 |
| 7 | 子类忘写 `__slots__` | `__dict__` 回归，内存优化失效 | 每层都声明 `__slots__` |
| 8 | `__getattr__` 内访问自身属性 | RecursionError | 用 `object.__getattribute__` |
| 9 | dataclass 可变默认值 | 类定义时直接 ValueError | `field(default_factory=list)` |

In [58]:
# ① 陷阱 1：类变量 + 可变对象 = 全局共享状态
class Bad:
    items = []                      # ② 类变量：所有实例共享同一个 list

    def add(self, x):
        self.items.append(x)        # ③ append 是原地修改，改的是类身上的那个 list

a, b = Bad(), Bad()
a.add(1)
print("症状: a 加了 1，b.items =", b.items, "← b 也变了！")

class Good:
    def __init__(self):
        self.items = []             # ④ 解法：实例属性，各用各的

    def add(self, x):
        self.items.append(x)

a2, b2 = Good(), Good()
a2.add(1)
print("解法: a2 加了 1，b2.items =", b2.items)

症状: a 加了 1，b.items = [1] ← b 也变了！
解法: a2 加了 1，b2.items = []


In [59]:
# ① 陷阱 2：定义 __eq__ 后 __hash__ 被隐式置 None
class OnlyEq:
    def __eq__(self, other):
        return isinstance(other, OnlyEq)   # ② 只定义了 __eq__

try:
    hash(OnlyEq())                         # ③ 哈希直接报错
except TypeError as e:
    print("症状:", type(e).__name__, ":", e)

class EqAndHash:
    def __eq__(self, other):
        return isinstance(other, EqAndHash)
    def __hash__(self):                    # ④ 解法：显式定义（不变对象基于同一组字段）
        return 42

print("解法: 可作字典键 =", {EqAndHash(): "v"}[EqAndHash()])

症状: TypeError : unhashable type: 'OnlyEq'
解法: 可作字典键 = v


In [60]:
# ① 陷阱 3：super() 不是「直接父类」——按 MRO 走
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B <- " + super().who()   # ② 直觉以为是 A
class C(A):
    def who(self): return "C <- " + super().who()
class D(B, C):
    def who(self): return "D <- " + super().who()

print("MRO:", [c.__name__ for c in D.__mro__])
print("调用链:", D().who())
# 症状：B 里的 super().who() 调到的是 C 而非 A（按 D 的 MRO 算）
# 解法：永远以「最终实例的 MRO」理解 super

MRO: ['D', 'B', 'C', 'A', 'object']
调用链: D <- B <- C <- A


In [61]:
# ① 陷阱 4：元类冲突
class Meta1(type): pass
class Meta2(type): pass

class C(metaclass=Meta1): pass

try:
    class D(C, metaclass=Meta2):    # ② 父类用 Meta1，自己声明 Meta2，且两者无继承关系
        pass
except TypeError as e:
    print("症状:", type(e).__name__, ":", e)

# ③ 解法一：让 Meta2 继承 Meta1（保持子类关系）
class Meta3(Meta1): pass
class E(C, metaclass=Meta3): pass
print("解法一: 正常创建", E.__name__)
# ④ 解法二（更推荐）：不用自定义元类，改用 __init_subclass__ / 类装饰器

症状: TypeError : metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases
解法一: 正常创建 E


In [62]:
# ① 陷阱 5：__del__ 不可靠 —— 资源释放要用 with
import gc

class BadResource:
    def __init__(self):
        self.file = open("/tmp/_oop_del_demo.txt", "w")   # ② 打开真实资源
        print("资源已打开")
    def __del__(self):
        self.file.close()                                  # ③ 期望在删除时关闭
        print("__del__ 已执行")

b = BadResource()
print("删除引用后 __del__ 是否立即执行？见下一行输出顺序：")
del b
gc.collect()                        # ④ 即使手动 GC，CPython 环境通常能触发，但【不构成保证】
print("程序继续运行（若上面没打印 __del__，即为症状；PyPy/循环引用下可能永不执行）")

# ⑤ 解法：上下文管理器，离开 with 块必然释放
class GoodResource:
    def __enter__(self):
        print("获取资源"); return self
    def __exit__(self, *exc):
        print("释放资源（必然执行）"); return False

with GoodResource():
    print("使用资源")

资源已打开
删除引用后 __del__ 是否立即执行？见下一行输出顺序：
__del__ 已执行
程序继续运行（若上面没打印 __del__，即为症状；PyPy/循环引用下可能永不执行）
获取资源
使用资源
释放资源（必然执行）


In [63]:
# ① 陷阱 6：子类用普通属性覆盖父类 property —— 静默失效
class A:
    @property
    def x(self):                    # ② 父类：x 是受控属性
        return 1

class B(A):
    x = 2                            # ③ 子类：类属性直接覆盖 property（不报错！）

print("症状: B().x =", B().x, "← 跳过了父类逻辑，且无任何警告")

class C(A):
    @property
    def x(self):                    # ④ 解法：子类覆盖也用 property
        return super().x * 10

print("解法: C().x =", C().x)

症状: B().x = 2 ← 跳过了父类逻辑，且无任何警告
解法: C().x = 10


In [64]:
# ① 陷阱 7：__getattr__ 无限递归
class Bad:
    def __getattr__(self, name):
        return self.name            # ② name 属性不存在 → 又触发 __getattr__ → 递归

try:
    Bad().x
except RecursionError:
    print("症状: RecursionError（无限递归）")

class Good:
    def __getattr__(self, name):
        if name == "name":                          # ③ 明确处理的属性
            return "fallback"
        raise AttributeError(f"没有属性 {name}")      # ④ 其余：正常抛错，不自我引用

g = Good()
print("解法: g.name =", g.name)
try:
    g.x
except AttributeError as e:
    print("解法: 未知属性 →", type(e).__name__, ":", e)

症状: RecursionError（无限递归）
解法: g.name = fallback
解法: 未知属性 → AttributeError : 没有属性 x


In [65]:
# ① 陷阱 9：dataclass 可变默认值（类定义时即报错，症状最「早」的一个）
from dataclasses import dataclass, field

try:
    @dataclass
    class Bad:
        items: list = []            # ② 直接给 list 字面量
except ValueError as e:
    print("症状:", type(e).__name__, ":", e)

@dataclass
class Good:
    items: list[str] = field(default_factory=list)   # ③ 解法：工厂函数

a, b = Good(), Good()
a.items.append(1)
print("解法: a.items =", a.items, "| b.items =", b.items, "← 互不影响")

症状: ValueError : mutable default <class 'list'> for field items is not allowed: use default_factory
解法: a.items = [1] | b.items = [] ← 互不影响


# 第 12 章 总结

## 核心机制速查表

| 机制 | 一句话作用 | 关键记忆点 |
|---|---|---|
| 魔术方法 | 让对象支持内置操作 | `__new__` 建对象、`__init__` 初始化 |
| `property` | 控制属性访问 | 只给 getter 即只读 |
| 描述符 | 可复用的属性逻辑 | 数据描述符 > 实例字典 > 非数据描述符 |
| MRO | 多继承方法查找顺序 | C3 线性化，`__mro__` 可查 |
| `super()` | 协作式调用 | 按 MRO 找「下一个」，不是父类 |
| Mixin | 功能复用 | 以 Mixin 结尾、放最左、`**kwargs` 协作 |
| ABC | 接口约束 | 抽象方法未实现 → 不能实例化 |
| 元类 | 控制类创建 | 优先 `__init_subclass__`，元类最后 |
| `dataclass` | 自动生成样板 | 可变默认值用 `default_factory` |
| `__slots__` | 限制属性 + 省内存 | 子类不声明则 `__dict__` 回归 |

## 通用代码模板（可直接复制）

```python
# 模板 1：类型校验描述符
class Typed:
    expected_type = object
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return obj.__dict__.get(self.name)
    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name} 必须是 {self.expected_type.__name__}")
        obj.__dict__[self.name] = value

class IntField(Typed): expected_type = int

# 模板 2：插件注册基类
class Plugin:
    registry = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.registry[cls.__name__] = cls

# 模板 3：协作式 Mixin 骨架
class SomeMixin:
    def __init__(self, *, own_param=None, **kwargs):
        super().__init__(**kwargs)     # 其余参数继续下传
        self.own_param = own_param

# 模板 4：上下文管理器
class Resource:
    def __enter__(self): return self
    def __exit__(self, exc_type, exc_val, exc_tb): return False

# 模板 5：不可变配置
from dataclasses import dataclass
@dataclass(frozen=True)
class Config:
    name: str
    retries: int = 3
```

## 记忆口诀

1. **查找顺序记心间**：数据描述符 > 实例字典 > 非数据描述符 > 类属性 > `__getattr__`
2. **定义 `__eq__` 别忘 `__hash__`**，且两者基于同一组字段
3. **`super()` 走 MRO 不是走父类**，多继承按最终实例算
4. **可变默认值两处坑**：类属性 list 共享、dataclass 用 `default_factory`
5. **资源释放靠 `with`，别信 `__del__`**
6. **元类最后用**：先 `__init_subclass__`，再类装饰器

## 自测练习（先自己写，再看参考实现）

1. **练习 1**：实现 `PositiveFloat` 数据描述符——赋值非正数时抛 `ValueError`，并用它建模 `temperature` 字段
2. **练习 2**：实现 `RetryMixin`（参数 `retries`，默认 3），与任意业务类组合，验证 `__mro__` 与协作式 `__init__`
3. **练习 3**：用 `@dataclass(frozen=True, order=True)` 实现 `Version`（major/minor），验证排序、相等、可哈希三件事
4. **练习 4**：给练习 3 的 `Version` 添加抽象基类 `Comparable`（含抽象属性 `summary`），并实现之
5. **练习 5**：实现一个 `Observable` 基类 + 两个观察者（打印日志 / 收集事件），模拟模型调用完成后的广播

In [66]:
# ① 练习 1 参考实现：PositiveFloat 数据描述符
class PositiveFloat:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.name)

    def __set__(self, obj, value):
        if not isinstance(value, float):
            raise TypeError(f"{self.name} 必须是 float")
        if value <= 0:
            raise ValueError(f"{self.name} 必须为正数")     # ② 范围校验
        obj.__dict__[self.name] = value

class Sampling:
    temperature = PositiveFloat()                            # ③ 声明式使用

    def __init__(self, temperature):
        self.temperature = temperature

s = Sampling(0.8)
print("合法:", s.temperature)
for bad in (1.5, -0.1):
    try:
        s.temperature = bad
    except (TypeError, ValueError) as e:
        print(f"赋值 {bad} →", type(e).__name__, ":", e)

合法: 0.8
赋值 -0.1 → ValueError : temperature 必须为正数


In [67]:
# ① 练习 2 参考实现：RetryMixin + 协作式多继承
class RetryMixin:
    def __init__(self, *, retries=3, **kwargs):   # ② 只认领 retries，其余下传
        super().__init__(**kwargs)
        self.retries = retries

class ServiceBase:
    def __init__(self, **kwargs):
        super().__init__(**kwargs)                # ③ 收口到 object
        self.started = False

class LLMClient(RetryMixin, ServiceBase):         # ④ 功能 + 基座自由组合
    def __init__(self, *, model, **kwargs):
        super().__init__(**kwargs)                # ⑤ 先走 RetryMixin
        self.model = model

c = LLMClient(model="gpt-4o", retries=5)
print("MRO:", [x.__name__ for x in LLMClient.__mro__])
print("参数各归其位:", c.model, c.retries, c.started)

MRO: ['LLMClient', 'RetryMixin', 'ServiceBase', 'object']
参数各归其位: gpt-4o 5 False


In [68]:
# ① 练习 3 参考实现：frozen + order 的 Version
from dataclasses import dataclass

@dataclass(frozen=True, order=True)               # ② order 生成 < <= > >=；frozen 生成 __hash__
class Version:
    major: int
    minor: int

vs = [Version(2, 0), Version(1, 9), Version(1, 2)]
print("排序:", [f"{v.major}.{v.minor}" for v in sorted(vs)])
print("相等:", Version(1, 2) == Version(1, 2))
print("可哈希:", {Version(1, 2): "v1.2"}[Version(1, 2)])
try:
    Version(1, 2).major = 9                       # ③ 不可变验证
except Exception as e:
    print("不可变:", type(e).__name__)

排序: ['1.2', '1.9', '2.0']
相等: True
可哈希: v1.2
不可变: FrozenInstanceError


In [69]:
# ① 练习 4 参考实现：ABC + 抽象属性 + dataclass 组合
from abc import ABC, abstractmethod
from dataclasses import dataclass

class Comparable(ABC):
    @property
    @abstractmethod
    def summary(self): ...                        # ② 抽象属性契约

@dataclass(frozen=True, order=True)
class Version(Comparable):
    major: int
    minor: int

    @property
    def summary(self):                            # ③ 履约：实现抽象属性
        return f"v{self.major}.{self.minor}"

v = Version(1, 2)
print("summary:", v.summary)
print("仍是 dataclass:", v == Version(1, 2), "|", v < Version(2, 0))

summary: v1.2
仍是 dataclass: True | True


In [70]:
# ① 练习 5 参考实现：观察者模式广播
class Observable:
    def __init__(self):
        self._observers = []

    def subscribe(self, observer):                # ② 观察者只需是可调用对象（__call__）
        self._observers.append(observer)

    def notify(self, event):
        for obs in self._observers:
            obs(event)

class ConsoleLogger:                              # ③ 观察者 A：打印
    def __call__(self, event):
        print("   [log]", event)

class EventCollector:                             # ④ 观察者 B：收集（用于测试断言）
    def __init__(self):
        self.events = []
    def __call__(self, event):
        self.events.append(event)

obs = Observable()
collector = EventCollector()
obs.subscribe(ConsoleLogger())
obs.subscribe(collector)

obs.notify("模型调用完成")                          # ⑤ 一次广播，多方接收
obs.notify("配额告警")
print("收集到的事件:", collector.events)

   [log] 模型调用完成
   [log] 配额告警
收集到的事件: ['模型调用完成', '配额告警']


## 结语

> Python 面向对象的高级之处不在于继承，而在于**数据模型协议**：通过魔术方法、描述符、元类，让自定义对象像内置对象一样自然地融入语言。

**下一步建议**（LLMOps 方向）：
1. 用 `pydantic` 重写第 10 章的参数校验组件，对比开发体验
2. 阅读 `functools.py` 中 `cached_property` 的真实源码（就是第 3 章非数据描述符）
3. 把 `LLMGateway` 扩展出 `RetryMixin` 与 `MetricsMixin`，验证协作式多继承的扩展性